### DTO 예제
- DTO(Data Transfer Object)를 설명할 때 가장 유명하고 이해하기 쉬운 예제가 바로 비밀번호 숨기기입니다.
- 초급자는 보통 이렇게 생각합니다. "DB에서 조회한 객체를 그대로 JSON으로 보내면 되는 것 아닌가?"
- 실무에서는 거의 그렇게 하지 않습니다.
- 왜냐하면 데이터베이스에는 외부에 공개하면 안 되는 정보가 포함되어 있기 때문입니다.

1. SQLite에 고객 정보 저장
2. Entity 객체 생성
3. Entity를 그대로 출력
4. DTO 생성
5. DTO로 변환
6. 비밀번호가 제거되는 것 확인

In [67]:
from dataclasses import dataclass, asdict
import sqlite3

In [68]:
# 데이터베이스 생성

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    password_hash TEXT NOT NULL
)
""")

conn.commit()
conn.close()

print("DB 생성 완료")

DB 생성 완료


In [69]:
# 샘플 데이터 저장

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

customer_list = [(
        "홍길동",
        "hong@test.com",
        "x7a9bc2de4f"
    ),
    (
        "김길동",
        "kim@test.com",
        "x7a8uc2de4f"
    )]

# cursor.execute(      # 한 행(row)을 입력하도록 설정
cursor.executemany(    # 여러 행을 한번에 입력하도록 설정 (260531)
    """
    INSERT INTO customers(
        name,
        email,
        password_hash
    )
    VALUES (?, ?, ?)
    """,
    customer_list
)

conn.commit()
conn.close()

print("데이터 저장 완료")

데이터 저장 완료


In [70]:
# 데이터 확인

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM customers
""")

rows = cursor.fetchall()

print(rows)

conn.close()

[(1, '홍길동', 'hong@test.com', 'x7a9bc2de4f'), (2, '김길동', 'kim@test.com', 'x7a8uc2de4f')]


In [71]:
# Entity 정의

@dataclass
class CustomerEntity:
    id: int
    name: str
    email: str
    password_hash: str

In [77]:
# DB → Entity 변환

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM customers
LIMIT 10      
""")

row = cursor.fetchall()   # 조회한 데이터를 저장 (2605331)
# print(row)
# row = cursor.fetchone()   # 조회한 데이터를 저장 (2605331)

conn.close()

customers = [
    CustomerEntity(     # 조회한 데이터를 Entity 변환
        id=row[0],
        name=row[1],
        email=row[2],
        password_hash=row[3]   # 비밀번호 : 숨겨야 할 내용 (260531)
    )
    for row in rows
]

print(customers)


[CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f'), CustomerEntity(id=2, name='김길동', email='kim@test.com', password_hash='x7a8uc2de4f')]


In [78]:
# Entity 그대로 출력

print(customers)   # 비밀번호가 그대로 노출되는 문제점 (260531)

[CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f'), CustomerEntity(id=2, name='김길동', email='kim@test.com', password_hash='x7a8uc2de4f')]


In [81]:
asdict(customers)    # json형태로 출력 

TypeError: asdict() should be called on dataclass instances

In [60]:
# DTO 정의

@dataclass
class CustomerResponseDTO:
    id: int
    name: str
    email: str

In [82]:
# Entity -> DTO 변환

response = CustomerResponseDTO(
    id=customers.id,
    name=customers.name,
    email=customers.email
)

response

AttributeError: 'list' object has no attribute 'id'

In [62]:
asdict(response)

{'id': (1, '홍길동', 'hong@test.com', 'x7a9bc2de4f'),
 'name': (2, '홍길동', 'hong@test.com', 'x7a9bc2de4f'),
 'email': (3, '김길동', 'kim@test.com', 'x7a8uc2de4f')}

In [100]:
customers

[CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f'),
 CustomerEntity(id=2, name='김길동', email='kim@test.com', password_hash='x7a8uc2de4f')]

In [ ]:
response_test = [
    CustomerResponseDTO(
        customer) for customer in customers
    ]
    
response_test      # 비밀번호를 전달받을 수 없으므로 에러 발생 (260531)

TypeError: CustomerResponseDTO.__init__() missing 2 required positional arguments: 'name' and 'email'

In [ ]:
from dataclasses import dataclass

@dataclass
class CustomerResponseDTO:
    id: int
    name: str
    email: str
    
    # Entity → DTO 변환이구나
    @classmethod                        # DTO 변환 메서드 추가 (260531) : 비밀번호만 제외하고 전달하도록 메서드 추가
    def from_entity(cls, customer):

        return cls(
            id=customer.id,
            name=customer.name,
            email=customer.email      # 비밀번호를 제외하고, 이메일까지만 전달 받음 (260531)
        )

In [ ]:
customers     # 먼저 customer 엔티티의 내용을 한번 더 확인 (260531)

[CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f'),
 CustomerEntity(id=2, name='김길동', email='kim@test.com', password_hash='x7a8uc2de4f')]

In [96]:
customers[1]

CustomerEntity(id=2, name='김길동', email='kim@test.com', password_hash='x7a8uc2de4f')

In [ ]:
response_test = [
    CustomerResponseDTO.from_entity(customer) for customer in customers    # 여러 개의 엔티티를 반복문을 이용하여 DTO 처리
]

response_test    # 비밀번호는 삭제되고 출력

[CustomerResponseDTO(id=1, name='홍길동', email='hong@test.com'),
 CustomerResponseDTO(id=2, name='김길동', email='kim@test.com')]

In [ ]:
# 여러개의 데이터를 처리하는 경우, 아래 코드는 잘못된 것을 확인할 것 (260531)

response_test =  CustomerResponseDTO.from_entity(customer)    # 변환 메서드를 사용하여 생성
response_test

CustomerResponseDTO(id=(1, '홍길동', 'hong@test.com', 'x7a9bc2de4f'), name=(2, '홍길동', 'hong@test.com', 'x7a9bc2de4f'), email=(3, '김길동', 'kim@test.com', 'x7a8uc2de4f'))

### DTO 변환 메서드에 @classmethod를 사용하는 이유는
1. 객체를 만들기 전에 호출할 수 있다.
2. cls(...) 형태로 새로운 객체를 생성할 수 있다.
3. 상속 구조에서도 올바르게 동작한다.
4. "Entity → DTO 변환 생성자"라는 의도를 명확히 표현한다.

- UserDTO.from_entity(user)
- OrderDTO.from_entity(order)
- CustomerDTO.from_model(customer)
- ProductDTO.from_entity(product)

- 이들은 모두 @classmethod를 사용한 대체 생성자 패턴의 대표적인 예입니다.